# BISINDO Final Experiment Control

**Impact of Vocabulary Expansion on Frame-Based BISINDO Recognition: A Controlled YOLO26 Study**

Notebook primer tunggal untuk alur final. Notebook lama tetap HISTORICAL EVIDENCE, bukan pesaing alur final.
**TASK 0: IMPLEMENTED; verifikasi melalui output nyata di bawah dan log pendamping.** TASK 1 diotorisasi dan audit selesai; TASK 2 diotorisasi untuk audit/usulan saja; TASK 3–8 belum diotorisasi.

Run All hanya inspeksi lokal: tidak training, inferensi, download, freeze, perubahan dataset, atau overwrite laporan.
Semua flag aksi default `False`; bahkan opt-in ditolak pada scaffold TASK 0. Tidak ada jalur scoring alternatif.
Hasil development tidak menjadi hasil external. Label VERIFIED hanya berlaku pada pemeriksaan yang benar-benar lulus;
COMPLETED hanya untuk task yang sudah memenuhi penerimaan, bukan karena bagiannya tersedia.

Bukti eksekusi TASK 0: `01_hasil/addendum/final_control_task0/`. Simpan output notebook setelah eksekusi.
Artefak ilmiah lama tetap sumber bukti; output notebook tidak menggantikannya.


## 1. Pertanyaan riset dan mapping beku

**Status: IMPLEMENTED.**

**Tujuan:** Mengukur perubahan common6 ketika vocabulary dan data diperluas 6 menjadi 8 kelas.

**Input/dependensi:** analysis_plan.json; mapping modul interference_metrics.

**Output:** Mapping dan seed tervalidasi.

**Kriteria penerimaan:** Mapping tepat, lima seed tetap, task classify.

**Risiko:** Frame recognition, bukan temporal translation; formal academic approval masih perlu konfirmasi.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata
import json
import platform
import sys

# Resolve from repository root, notebook folder, or the existing compatibility runtime.
candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
roots = [p for p in candidates if (p/'05_kode/src/confirmatory_evaluation.py').is_file()]
if not roots:
    raise FileNotFoundError('Open notebook inside the BISINDO repository')
ROOT = roots[0]
sys.path.insert(0, str(ROOT/'05_kode/src'))
import confirmatory_evaluation as evaluation
import confirmatory_statistics as statistics
import interference_metrics as interference
import classification_research as research

BASE = ['saya', 'kamu', 'senang', 'bingung', 'marah', 'apa_kabar']
EXTRA = ['halo', 'tidur']
ALL_CLASSES = BASE + EXTRA
SEEDS = [42, 43, 44, 45, 46]
PLAN_PATH = ROOT/'07_dokumen/protokol/analysis_plan.json'
PLAN = evaluation.read(PLAN_PATH)
assert PLAN['class_mapping'] == ALL_CLASSES == list(interference.ALL_CLASSES)
assert PLAN['training_seeds'] == SEEDS
assert PLAN['task'] == 'classify' and PLAN['model_variant'] == 'yolo26n-cls'
RUN_TRAINING = False
RUN_FREEZE = False
RUN_EXTERNAL_SCORING = False
print({'status': 'VERIFIED', 'classes': ALL_CLASSES, 'seeds': SEEDS,
       'task': PLAN['task'], 'primary_metric': 'paired common6 macro-F1 B minus A; B full8 argmax'})

{'status': 'VERIFIED', 'classes': ['saya', 'kamu', 'senang', 'bingung', 'marah', 'apa_kabar', 'halo', 'tidur'], 'seeds': [42, 43, 44, 45, 46], 'task': 'classify', 'primary_metric': 'paired common6 macro-F1 B minus A; B full8 argmax'}


## 2. Environment, Git, dan provenance

**Status: IMPLEMENTED.**

**Tujuan:** Mencatat environment aktual dan hash sumber tanpa memuat model.

**Input/dependensi:** Interpreter aktif; metadata distribusi; Git lokal; file protokol.

**Output:** Waktu UTC, software, host, revisi, dirty tree, hash sumber/config.

**Kriteria penerimaan:** Dependensi inspeksi tersedia; status Git diungkapkan.

**Risiko:** Versi runtime saat audit bukan otomatis environment training historis; working tree memuat file untracked awal.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

In [2]:
AUDIT_STARTED_UTC = datetime.now(timezone.utc).isoformat()
ENVIRONMENT = {'python': sys.version, 'executable': sys.executable,
               'platform': platform.platform(),
               'packages': {name: metadata.version(name) for name in
                            ['nbformat', 'nbclient', 'ipykernel', 'numpy', 'pandas', 'scipy', 'torch', 'torchvision', 'ultralytics']},
               'git': evaluation.git_state(ROOT)}
SOURCE_HASHES = {str(p.relative_to(ROOT)): evaluation.sha(p)
                 for p in sorted((ROOT/'05_kode/src').glob('*.py'))}
print(json.dumps({'started_utc': AUDIT_STARTED_UTC, 'environment': ENVIRONMENT,
                  'source_sha256': SOURCE_HASHES, 'plan_sha256': evaluation.sha(PLAN_PATH)}, indent=2))

{
  "started_utc": "2026-11-05T06:13:30.519102+00:00",
  "environment": {
    "python": "3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]",
    "executable": "C:\\Users\\Ricardo Cuthbert\\Downloads\\skrpsi bwabwa\\.venv-bisindo\\Scripts\\python.exe",
    "platform": "Windows-10-10.0.26300-SP0",
    "packages": {
      "nbformat": "5.10.4",
      "nbclient": "0.10.0",
      "ipykernel": "6.29.4",
      "numpy": "1.26.4",
      "pandas": "2.2.3",
      "scipy": "1.17.1",
      "torch": "2.14.0+cu126",
      "torchvision": "0.29.0+cu126",
      "ultralytics": "8.4.197"
    },
    "git": {
      "commit": "89df4d256a26ad1de3828ea24d6319f42da8f1ee",
      "working_tree": "M 06_notebook/README.md\n M 07_dokumen/TRACKING_BISINDO.md\n?? 01_hasil/addendum/\n?? 05_kode/src/confirmatory_evaluation.py\n?? 05_kode/src/confirmatory_statistics.py\n?? 05_kode/src/interference_metrics.py\n?? 05_kode/tests/test_confirmatory_statistics.py\n?? 05_kode/tests/test_final_contr

## 3. Inventaris data dan split

**Status: IMPLEMENTED / BLOCKED final manifests.**

**Tujuan:** Menemukan artefak split dan manifest yang harus diverifikasi TASK 1.

**Input/dependensi:** Path plan, split_lock.csv, development manifest.csv; tidak membuka external predictions.

**Output:** Keberadaan dan SHA-256 file metadata; daftar input final kosong.

**Kriteria penerimaan:** Path artefak existing valid; null dilaporkan sebagai missing, tidak diisi tebakan.

**Risiko:** Hash metadata bukan pemeriksaan seluruh byte video, leakage, signer, atau validitas label.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

**TASK 1:** technical audit VERIFIED; final readiness BLOCKED. Historical TASK 0 outputs below remain unchanged. Current findings follow in a separate cell.

In [3]:
ARTIFACT_INVENTORY = {}
for name, value in PLAN['artifacts'].items():
    if value is None:
        ARTIFACT_INVENTORY[name] = {'status': 'BLOCKED', 'path': None}
    else:
        path = evaluation.resolve(ROOT, value)
        if not path.is_file():
            raise FileNotFoundError(path)
        ARTIFACT_INVENTORY[name] = {'status': 'VERIFIED existence/hash only', 'path': value,
                                    'sha256': evaluation.sha(path)}
print(json.dumps(ARTIFACT_INVENTORY, indent=2))
print('TASK 1 PLANNED: count/schema/split/duplicate/exposure checks; not executed by this cell.')

{
  "addendum": {
    "status": "VERIFIED existence/hash only",
    "path": "07_dokumen/protokol/HYPOTHESES_ENDPOINTS_ADDENDUM.md",
    "sha256": "a0b557615af7cac04592f72de6a1ad37f90f35bfd8715121399087b47b550ea2"
  },
  "experiment_config": {
    "status": "BLOCKED",
    "path": null
  },
  "initial_checkpoint": {
    "status": "VERIFIED existence/hash only",
    "path": "02_exp/arsip_pipeline/experiment_yolo26_classification_pilot/checkpoints/yolo26n-cls.pt",
    "sha256": "0dd6f8dbc448870ac98a3cbb7156f923f7ce21fed3755d4019169ffffd279e81"
  },
  "dataset_manifest": {
    "status": "VERIFIED existence/hash only",
    "path": "02_exp/arsip_pipeline/experiment_yolo26_v3/split_lock.csv",
    "sha256": "03eb34d91c6764d14052d2d8f99f6996d5b54602d50d533b47b9246ad32f706e"
  },
  "split_mapping": {
    "status": "VERIFIED existence/hash only",
    "path": "02_exp/arsip_pipeline/experiment_yolo26_v3/split_lock.csv",
    "sha256": "03eb34d91c6764d14052d2d8f99f6996d5b54602d50d533b47b9246ad32f706e"

### TASK 1 — hasil audit file aktual

**Status: VERIFIED technical audit / BLOCKED final readiness.** Input: immutable `audit.json` dan `provenance.json` hasil audit read-only.
Output: expected/observed counts, pairing lima seed, exact duplicate findings dan provenance evidence table.
Penerimaan: file physical/hashes diperiksa; unknown dinyatakan UNVERIFIED; tidak training, scoring, selection, atau perubahan split.
Risiko: exact hashes tidak membuktikan signer/semantic independence; near-duplicate dan all-frame raw video comparison belum dilakukan.
Sel ini hanya membaca evidence; tidak menjalankan ulang auditor, model, atau frame extraction.

[Laporan TASK 1](../../01_hasil/addendum/data_inventory_task1_20261105/audit_report.md) ·
[Audit JSON](../../01_hasil/addendum/data_inventory_task1_20261105/audit.json) ·
[Provenance](../../01_hasil/addendum/data_inventory_task1_20261105/provenance.json).
Outputs sel lama adalah catatan eksekusi TASK 0, tidak diganti dengan hasil eksekusi baru.


In [4]:
TASK1_DIRECTORY = ROOT/'01_hasil/addendum/data_inventory_task1_20261105'
TASK1_AUDIT = evaluation.read(TASK1_DIRECTORY/'audit.json')
TASK1_PROVENANCE = evaluation.read(TASK1_DIRECTORY/'provenance.json')
TASK1_DUPLICATES = evaluation.read(TASK1_DIRECTORY/'duplicate_impact.json')
assert TASK1_AUDIT['schema_version'] == 1 and TASK1_AUDIT['task'] == 1
assert TASK1_AUDIT['technical_audit_passed'] is True
assert TASK1_AUDIT['class_mapping'] == ALL_CLASSES
assert [row['seed'] for row in TASK1_AUDIT['internal_pairing']] == SEEDS
print(json.dumps({'task': 1, 'status': 'VERIFIED read-only audit; external readiness BLOCKED',
                  'audit_completed_utc': TASK1_AUDIT['completed_utc'],
                  'expected': TASK1_AUDIT['expected'], 'observed': TASK1_AUDIT['observed'],
                  'class_split_counts': TASK1_AUDIT['class_split_counts'],
                  'pairing': TASK1_AUDIT['internal_pairing'],
                  'frame_provenance': TASK1_AUDIT['frame_provenance'],
                  'actual_video_duplicate_groups': len(TASK1_AUDIT['duplicates']['actual_video_groups']),
                  'decoded_rgb_duplicate_groups': len(TASK1_DUPLICATES['groups']),
                  'cross_split_groups': TASK1_DUPLICATES['cross_split_groups'],
                  'duplicate_interpretation': TASK1_DUPLICATES['interpretation'],
                  'near_duplicate_status': TASK1_AUDIT['duplicates']['near_duplicates'],
                  'final_manifests': TASK1_AUDIT['final_manifest_paths'],
                  'provenance': TASK1_PROVENANCE['evidence_table'],
                  'audit_sha256': evaluation.sha(TASK1_DIRECTORY/'audit.json'),
                  'provenance_sha256': evaluation.sha(TASK1_DIRECTORY/'provenance.json')}, indent=2))


{
  "task": 1,
  "status": "VERIFIED read-only audit; external readiness BLOCKED",
  "audit_completed_utc": "2026-11-05T09:12:49.827514+00:00",
  "expected": {
    "internal": 2400,
    "external": 400,
    "internal_train": 1920,
    "internal_val": 480,
    "external_common6": 300,
    "external_full8": 400
  },
  "observed": {
    "inventory_rows": 2800,
    "internal": 2400,
    "external": 400,
    "internal_train": 1920,
    "internal_val": 480,
    "external_common6": 300,
    "external_full8": 400,
    "physical_video_files": 2800,
    "physical_verified_files": 2800
  },
  "class_split_counts": [
    {
      "source": "test",
      "split": "test",
      "class_name": "apa_kabar",
      "n": 50
    },
    {
      "source": "test",
      "split": "test",
      "class_name": "bingung",
      "n": 50
    },
    {
      "source": "test",
      "split": "test",
      "class_name": "halo",
      "n": 50
    },
    {
      "source": "test",
      "split": "test",
      "class_name": 

## 4. Frame selection dan preprocessing

**Status: IMPLEMENTED inspection / PLANNED final verification.**

**Tujuan:** Menelusuri recipe tanpa memilih ulang frame.

**Input/dependensi:** classification_pilot.letterbox, frame_candidates/pilot_study; Stage1b config/source_snapshot.

**Output:** Recipe tertulis dan hash config/source snapshot.

**Kriteria penerimaan:** Common6 frame/pixels harus identik pada final release; verifikasinya TASK 1/3.

**Risiko:** Training override Resize+ToTensor+identity Normalize berbeda dari default crop Ultralytics; audit effective transforms TASK 2.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

In [4]:
WORKS = {seed: evaluation.resolve(ROOT, PLAN['development_runs'][str(seed)]) for seed in SEEDS}
REFERENCE_CONFIG = evaluation.read(WORKS[42]/'reports/experiment_config.json')
print(json.dumps({'preprocessing_recorded': REFERENCE_CONFIG['preprocessing'],
                  'evaluation_checkpoint': REFERENCE_CONFIG['evaluation_checkpoint'],
                  'source_snapshot_recorded': REFERENCE_CONFIG['source_snapshot'],
                  'config_sha256': evaluation.sha(WORKS[42]/'reports/experiment_config.json'),
                  'frame_policy_source': ['05_kode/src/frame_candidates.py', '05_kode/src/pilot_study.py',
                                          '05_kode/src/classification_pilot.py'],
                  'final_frame_provenance_status': 'BLOCKED: final manifests unresolved'}, indent=2))

{
  "preprocessing_recorded": "Full-frame aspect-preserving letterbox to 224x224, gray114; RGB/255; no crop/flip",
  "evaluation_checkpoint": "last.pt after 30 epochs, same for A/B",
  "source_snapshot_recorded": {
    "split_lock.csv": "03eb34d91c6764d14052d2d8f99f6996d5b54602d50d533b47b9246ad32f706e",
    "candidates.csv": "2061a83fa605b0194324499ba7f9b9f69ddfd21ca86a907db01786fdcb3afb12",
    "candidate_config.json": "eb986c300d2d618372f955e87565f7f32c8abbfd3d7b3a34f2e229361f0eb618",
    "human_pilot_manifest": "6d033a09eb54c5f0af011f3b7fecf7fe5a9b2e5532610c9d8468961ae43128fd"
  },
  "config_sha256": "60b74d1b0511a49b51123a31d13df9983aa08034fb2aac8bcf25a4d56f1baa2c",
  "frame_policy_source": [
    "05_kode/src/frame_candidates.py",
    "05_kode/src/pilot_study.py",
    "05_kode/src/classification_pilot.py"
  ],
  "final_frame_provenance_status": "BLOCKED: final manifests unresolved"
}


## 5. Konfigurasi training dan seed

**Status: IMPLEMENTED audit / PLANNED final recipe.**

**Tujuan:** Memeriksa sepuluh checkpoint development tanpa deserialisasi weights.

**Input/dependensi:** trained_models.json, model_*_execution.json, experiment_config.json, results.csv.

**Output:** Hash checkpoint aktual, effective args, steps, durasi tercatat; histori seed dan failure.

**Kriteria penerimaan:** Semua pasangan 42–46 dipertahankan; hash cocok; initialization pinned independen tercatat.

**Risiko:** Bukti source/records bukan replay training; lima seed bukan seluruh stochasticity; tidak memilih seed terbaik.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

**Provenance seed sementara (bukan audit historis lengkap TASK 2):** alasan awal memilih angka 42 tidak ditemukan.
`PREREGISTRATION.md` 2026-11-03 sudah menetapkan 42–46; Addendum01 menyatakan hanya pilot seed42 telah diamati.
`DEVIATION_LOG.md` D002 menyatakan pertanyaan/protokol baru muncul setelah hasil seed42 diketahui.
Dengan demikian jadwal lima seed bukan prospektif terhadap pilot pertama. Tanggal/urutan persetujuan berasal dari records,
bukan bukti chat independen. Persetujuan Stage1b per seed bertanggal 2026-11-03. Waktu tepat penambahan 43–46 belum dipastikan.
Tidak ada alasan statistik khusus bagi angka 42–46. Lima paired seed memberi penilaian praktis variabilitas dengan keterbatasan compute.

Kegagalan yang ditemukan: runner preflight v1 (0 fit), probe A seed42 attempt1 (gagal sebelum optimization),
notebook multi-work dalam satu proses (Ultralytics settings). Semua artefak lama dipertahankan.
Kelengkapan semua run yang pernah dicoba belum dibuktikan; jangan menyatakan tidak ada run terhilang.


In [5]:
import pandas as pd
RUN_ROWS = []
for seed, work in WORKS.items():
    config = evaluation.read(work/'reports/experiment_config.json')
    trained = evaluation.read(work/'runs/trained_models.json')
    assert {x['model'] for x in trained['models']} == {'A', 'B'} and len(trained['models']) == 2
    for item in trained['models']:
        model = item['model']
        checkpoint = work/'runs'/f'model_{model}'/'weights/last.pt'
        execution = evaluation.read(work/'runs'/f'model_{model}_execution.json')
        curve = pd.read_csv(work/'runs'/f'model_{model}'/'results.csv')
        curve.columns = curve.columns.str.strip()
        actual_sha = evaluation.sha(checkpoint)
        assert actual_sha == item['sha256']
        assert item['seed'] == seed and item['epoch'] == 30
        assert item['initial_checkpoint_sha256'] == execution['initial_checkpoint_sha256'] == research.PINNED_SHA
        assert curve.epoch.tolist() == list(range(1, 31))
        args = execution['actual_args']
        for key in ['seed', 'epochs', 'imgsz', 'batch', 'optimizer']:
            assert args[key] == config['training'][key]
        RUN_ROWS.append({'seed': seed, 'model': model, 'tier': 'development',
                         'checkpoint': str(checkpoint.relative_to(ROOT)), 'sha256': actual_sha,
                         'optimizer_steps': execution['actual_optimizer_steps'],
                         'recorded_training_seconds': float(curve.time.iloc[-1]),
                         'effective_args': args})
print(pd.DataFrame([{k:v for k,v in row.items() if k != 'effective_args'} for row in RUN_ROWS]).to_string(index=False))
print('Effective historical arguments:')
print(json.dumps(RUN_ROWS, indent=2))
FAILURE_PATH = ROOT/'01_hasil/tahap1b/exploratory_development_stage1b_agent_execution_20261103_v2/exploratory_failures.json'
print('Historical failures:', json.dumps(evaluation.read(FAILURE_PATH), indent=2))

 seed model        tier                                                                                                    checkpoint                                                           sha256  optimizer_steps  recorded_training_seconds
   42     A development 02_exp\tahap1b\full_runs\exploratory_development_stage1b_user_20261103_v1_seed42\runs\model_A\weights\last.pt 03727737a647405bdc6154c925c4575f2b77bf45f48a2ef21a92d02d044f54b7              690                    53.1245
   42     B development 02_exp\tahap1b\full_runs\exploratory_development_stage1b_user_20261103_v1_seed42\runs\model_B\weights\last.pt e544a04b4798f1d76a2096c844fd768bd53e50aa2fa4e1c17a419eef3fbc67bd              900                    67.1860
   43     A development 02_exp\tahap1b\full_runs\exploratory_development_stage1b_user_20261103_v1_seed43\runs\model_A\weights\last.pt 20533ef76136e29f105f96bcd9aeda09d530a2bcc65198beaf94bf9df4ebe5e7              690                    54.3363
   43     B development 02_e

### TASK 2 — effective recipe dan seed (2026-11-09)

**Status: audit tersedia / kandidat PROPOSED_NOT_APPROVED / final readiness BLOCKED.**
Checkpoint metadata + source override + args/log/counters/results mendukung recipe efektif.
Tidak ada historical batch-level transform trace; rekonstruksi source framework lokal bukan replay training.
TASK 0/1 outputs di atas adalah bukti historis dan dipertahankan.


In [7]:
TASK2_DIRECTORY = ROOT/'01_hasil/addendum/scientific_decisions_task2_20261109'
TASK2_AUDIT = evaluation.read(TASK2_DIRECTORY/'effective_configuration.json')
TASK2_CANDIDATE = evaluation.read(TASK2_DIRECTORY/'candidate_config.json')
TASK2_SEEDS = evaluation.read(TASK2_DIRECTORY/'seed_provenance.json')
assert TASK2_CANDIDATE['status'] == 'PROPOSED_NOT_APPROVED'
assert TASK2_CANDIDATE['training_authorized'] is False
display(pd.DataFrame([{'seed': row['seed'], 'arm': row['arm'],
                      'epochs': row['configured']['epochs'], 'batch': row['configured']['batch'],
                      'steps': row['steps'], 'epoch30_lr': row['epoch30_learning_rates'][0],
                      'transforms': row['checkpoint_transform']}
                     for row in TASK2_AUDIT['runs']]))
print({'candidate_status': TASK2_CANDIDATE['status'], 'training': TASK2_CANDIDATE['training'],
       'planned_arms': TASK2_CANDIDATE['arms'], 'deviations': TASK2_CANDIDATE['deviations_from_stage1b'],
       'seed_provenance': TASK2_SEEDS, 'evidence_limits': TASK2_AUDIT['evidence_limits']})


,seed,arm,epochs,batch,steps,epoch30_lr,transforms
0,42,A,30,16,690,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."
1,42,B,30,16,900,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."
2,43,A,30,16,690,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."
3,43,B,30,16,900,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."
4,44,A,30,16,690,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."
5,44,B,30,16,900,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."
6,45,A,30,16,690,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."
7,45,B,30,16,900,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."
8,46,A,30,16,690,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."
9,46,B,30,16,900,0.00043,"Compose(\n Resize(size=(224, 224), interpol..."


{'candidate_status': 'PROPOSED_NOT_APPROVED', 'training': {'epochs': 30, 'imgsz': 224, 'batch': 16, 'deterministic': True, 'optimizer': 'SGD', 'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'warmup_bias_lr': 0.1, 'cos_lr': False, 'patience': 0, 'nbs': 16, 'device': 0, 'workers': 0, 'amp': False, 'compile': False, 'pretrained': True, 'resume': False, 'single_cls': False, 'fraction': 1.0, 'cache': False, 'rect': False, 'multi_scale': 0.0, 'degrees': 0.0, 'translate': 0.0, 'scale': 0.0, 'shear': 0.0, 'perspective': 0.0, 'fliplr': 0.0, 'flipud': 0.0, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'bgr': 0.0, 'mosaic': 0.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 0, 'erasing': 0.0, 'auto_augment': None, 'val': True, 'save': True, 'plots': True, 'exist_ok': False, 'verbose': True, 'task': 'classify', 'dropout': 0.0}, 'planned_arms': {'A': {'class_ids': [0, 1, 2, 3, 4, 5], 'planned_train': 1440, 'planned_va

## 6. Controller training Model A/B

**Status: PLANNED / BLOCKED.**

**Tujuan:** Reuse research.prepare/train dan gate existing, bukan pipeline baru.

**Input/dependensi:** Settings, split/frame release final, pretrained lokal, scoped approval fase final.

**Output:** Pada TASK 4: sepuluh run dengan log, config, hash, timestamps, exit code, duration, steps.

**Kriteria penerimaan:** A/B masing-masing mulai yolo26n-cls.pt; tidak A ke B; fresh subprocess per run; tidak resume/retry otomatis.

**Risiko:** authorize_training existing hanya scope stage1b: belum merupakan otorisasi final. Jangan memberi label final ke development.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

In [6]:
if RUN_TRAINING:
    raise PermissionError('TASK 0: final training controller not authorized/implemented; LANJUT TASK 3/4 and scoped approval required')
print('SKIPPED training: RUN_TRAINING=False; zero model fits')

SKIPPED training: RUN_TRAINING=False; zero model fits


## 7. Internal validation dan development metrics

**Status: IMPLEMENTED existing-report inspection.**

**Tujuan:** Membaca laporan internal yang sudah ada, tanpa inferensi atau bootstrap baru.

**Input/dependensi:** development_interference_20261105/common6_primary_results.csv dan analysis_details.json.

**Output:** Nilai tersimpan berlabel development; file hash sebagai provenance.

**Kriteria penerimaan:** N120 common6, B N160, lima seed; bukan external/final result.

**Risiko:** TASK 0 memverifikasi isi laporan saja; recompute predictions/alignment/statistics menunggu TASK 5.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

In [7]:
DEVELOPMENT = ROOT/'01_hasil/addendum/development_interference_20261105'
summary = pd.read_csv(DEVELOPMENT/'common6_primary_results.csv', keep_default_na=False)
required = {'seed', 'n', 'macro_f1_A', 'macro_f1_B', 'delta', 'evidence_tier'}
if not required.issubset(summary.columns):
    raise ValueError('Existing development summary schema changed')
assert set(summary.evidence_tier) == {'development'}
assert set(summary.loc[summary.seed.ne('mean'), 'seed'].astype(int)) == set(SEEDS)
details = evaluation.read(DEVELOPMENT/'analysis_details.json')
assert details['evidence_tier'] == 'development' and details['training_seeds'] == SEEDS
print(summary.to_string(index=False))
print({'recorded_common6_n': details['n_videos'], 'recorded_full8_n': details['n_full8_videos'],
       'ni_decision': details['ni_decision'],
       'report_sha256': evaluation.sha(DEVELOPMENT/'common6_primary_results.csv')})

seed   n  macro_f1_A  macro_f1_B     delta evidence_tier              ci95_low            ci95_high    lower_one_sided95    upper_one_sided95         bootstrap_se                      ni_decision
  42 120    0.931776    0.930042 -0.001734   development                                                                                                                                           
  43 120    0.932625    0.930894 -0.001731   development                                                                                                                                           
  44 120    0.947500    0.931947 -0.015553   development                                                                                                                                           
  45 120    0.956626    0.923807 -0.032819   development                                                                                                                                           
  46 120    0.930972

## 8. Keputusan ilmiah dan external readiness

**Status: IMPLEMENTED / BLOCKED.**

**Tujuan:** Mengungkap konflik tanpa mengubah hipotesis.

**Input/dependensi:** Preregistration, Addendum01, draft addendum, analysis plan, readiness report.

**Output:** Status unresolved dan expected refusal validator existing.

**Kriteria penerimaan:** Tidak mengganti H1/H2; academic approval dan exposure review dibuktikan sebelum final.

**Risiko:** H1 .02 historis belum punya justifikasi substantif; signer/exposure belum lengkap; judul master berbeda dari Addendum01.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

H2 dipertahankan verbatim dari `PREREGISTRATION.md` bagian 1:

- H2 kemampuan baru: accuracy B pada GT halo **dan** GT tidur masing-masing ≥ **0.80**.
  Ambang 80% per kelas disetujui peneliti pada 2026-11-03, sebelum run baru.
  Laporkan point estimate dan CI per kelas. Aturan konfirmatori yang diusulkan: lower one-sided 95% CI
  masing-masing kelas > 0.80 (intersection-union: kedua komponen harus lolos).
  Point estimate ≥ 0.80 saja disebut memenuhi target deskriptif, bukan lulus pengujian konfirmatori.

H1: `NI_MARGIN_UNRESOLVED`; margin aktif null. Perlu margin berjustifikasi atau amendment estimation-only yang disetujui.
Academic approval Classification, judul final, recipe final, signer provenance, exposure Kaggle, final manifests dan sepuluh final checkpoints belum selesai.
Kaggle pernah dipreprocess. Tidak menyatakan untouched atau signer-independent.


In [8]:
print({'plan_status': PLAN['status'], 'ni_status': PLAN['ni_status'],
       'h2_status': PLAN['h2_status'], 'external_readiness': 'BLOCKED'})
# An expected refusal is evidence of the gate, not a successful final validation.
if PLAN['status'] != 'final':
    try:
        evaluation.validate_plan(PLAN, ROOT)
    except ValueError as error:
        if str(error) != 'Final Classification configuration unresolved':
            raise
        print('VERIFIED expected refusal:', str(error))
    else:
        raise AssertionError('Draft plan unexpectedly accepted')
else:
    print('Final plan present: complete readiness checks require separately approved TASK 6')

{'plan_status': 'draft_blocked', 'ni_status': 'NI_MARGIN_UNRESOLVED', 'h2_status': 'preserved_compatible', 'external_readiness': 'BLOCKED'}
VERIFIED expected refusal: Final Classification configuration unresolved


### TASK 2 — keputusan ilmiah menunggu persetujuan

Rekomendasi H1: **Option B**, estimasi paired mean Δmacro-F1 dan two-sided95% hierarchical CI.
Ini usulan, bukan amendemen. Option A memerlukan justifikasi substantif δ yang belum tersedia.
H2 verbatim pada sel historis tetap berlaku. Approval akademik Classification/judul/pertanyaan riset belum terbukti.
Duplikasi frame7/10/14 senang223/224 adalah tiga pasangan frame dari dua video locked train;
bukan tiga pasangan video. Development hanya memakai224/frame48. Content-cluster review tetap pending.
Kaggle provenance/license, signer/session/exposure attestation, final manifests dan semantic labels belum lengkap.
Laporan lengkap: `01_hasil/addendum/scientific_decisions_task2_20261109/report.md`.


In [11]:
TASK2_DECISIONS = evaluation.read(TASK2_DIRECTORY/'scientific_decisions.json')
assert TASK2_DECISIONS['protocol_amended'] is False
assert TASK2_DECISIONS['task3_authorized'] is False
print(TASK2_DECISIONS)
print({'final_readiness': TASK2_CANDIDATE['readiness'],
       'TASK3': 'NOT_AUTHORIZED', 'training_scoring_freeze': 'DISABLED'})


{'status': 'PROPOSED_NOT_APPROVED', 'recommended_h1': 'B', 'approved_h1': None, 'protocol_amended': False, 'h1_options': {'A': 'NI only after substantive outcome-independent delta justification and approval', 'B': 'mean paired macro-F1 delta with two-sided95% hierarchical CI; no NI success claim'}, 'h2_verbatim': '- H2 kemampuan baru: accuracy B pada GT halo **dan** GT tidur masing-masing ≥ **0.80**.\n  Ambang 80% per kelas disetujui peneliti pada 2026-11-03, sebelum run baru.\n  Laporkan point estimate dan CI per kelas. Aturan konfirmatori yang diusulkan: lower one-sided 95% CI\n  masing-masing kelas > 0.80 (intersection-union: kedua komponen harus lolos).\n  Point estimate ≥ 0.80 saja disebut memenuhi target deskriptif, bukan lulus pengujian konfirmatori.', 'h2_threshold': 0.8, 'h2_success': 'both lower one-sided95% bounds >0.80; equality fails', 'academic_approval_evidence': None, 'signatures': None, 'finalization_blocked': True, 'task3_authorized': False}
{'final_readiness': {'stat

## 9. Final freeze dan integrity

**Status: PLANNED / BLOCKED.**

**Tujuan:** Reuse evaluation.freeze setelah semua gates selesai.

**Input/dependensi:** Final plan, frame manifests, 10 final checkpoints, synthetic tests, environment/reviewer attestations.

**Output:** TASK 6: immutable freeze dan SHA-256 aktual untuk approval.

**Kriteria penerimaan:** Sumber/input byte terikat; missing evidence tidak dilewati; output baru eksklusif.

**Risiko:** Jangan mempromosikan checkpoint Stage1b atau inventory audit menjadi final freeze.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

In [9]:
if RUN_FREEZE:
    raise PermissionError('TASK 0: freeze disabled; complete approved TASK 6 first')
print('SKIPPED freeze: RUN_FREEZE=False; no scientific artifacts frozen')

SKIPPED freeze: RUN_FREEZE=False; no scientific artifacts frozen


## 10. External scoring bergate

**Status: PLANNED / BLOCKED.**

**Tujuan:** Hanya evaluation.score CLI existing; tidak ada inference alternatif.

**Input/dependensi:** Freeze valid dan approval reviewer terikat exact freeze SHA-256; output baru.

**Output:** TASK 7: satu batch semua lima seed/dua model, raw predictions, receipts, logs, hashes.

**Kriteria penerimaan:** verify_freeze/validate_plan existing tetap berlaku; common6/full8 sesuai protocol.

**Risiko:** No automatic retry. Jangan menghapus scoring_started receipt setelah gagal. Boolean saja tidak memberi izin.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

In [10]:
if RUN_EXTERNAL_SCORING:
    raise PermissionError('TASK 0: scoring disabled; TASK 7 requires approval bound to validated freeze hash')
print('SKIPPED external scoring: RUN_EXTERNAL_SCORING=False; no external predictions accessed')

SKIPPED external scoring: RUN_EXTERNAL_SCORING=False; no external predictions accessed


Interface yang telah ditemukan (dokumentasi, **tidak dieksekusi**):

```powershell
# TASK 6 setelah izin dan artefak lengkap, gunakan path output baru:
& .venv-bisindo/Scripts/python.exe 05_kode/src/confirmatory_evaluation.py freeze --plan 07_dokumen/protokol/analysis_plan.json --out PATH_TO_NEW_FREEZE.json
# TASK 7 setelah approval exact freeze hash; sekali per frozen batch:
& .venv-bisindo/Scripts/python.exe 05_kode/src/confirmatory_evaluation.py score --freeze PATH_TO_NEW_FREEZE.json --approval PATH_TO_REAL_APPROVAL.json --out PATH_TO_NEW_RESULTS
```

Placeholder bukan approval. Controller runnable TASK 0 sengaja tidak memiliki jalur aksi. TASK 3/6/7 harus melengkapi bagian ini setelah izin.


## 11. Statistik paired dan interference

**Status: PLANNED final / IMPLEMENTED module reuse.**

**Tujuan:** Reuse interference.interference dan statistics.bootstrap; tidak menyalin algoritme.

**Input/dependensi:** Prediksi paired yang tervalidasi, full8 argmax B, semua seed tetap.

**Output:** TASK 5/8: delta F1, interval hierarchical, per-class precision/recall/F1, I1/I2/I3, full8.

**Kriteria penerimaan:** Pair video/GT/frame sama; seed-video bukan observasi independen; tidak memilih seed/performance.

**Risiko:** Restricted common6 argmax hanya diagnostik; estimation internal bukan konfirmasi H1/H2.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

In [11]:
print({'metrics_module': str(Path(interference.__file__).relative_to(ROOT)),
       'statistics_module': str(Path(statistics.__file__).relative_to(ROOT)),
       'evaluation_module': str(Path(evaluation.__file__).relative_to(ROOT)),
       'functions': [interference.interference.__name__, statistics.bootstrap.__name__, evaluation.export_reports.__name__],
       'status': 'IMPLEMENTED in existing modules; final analysis PLANNED'})

{'metrics_module': '05_kode\\src\\interference_metrics.py', 'statistics_module': '05_kode\\src\\confirmatory_statistics.py', 'evaluation_module': '05_kode\\src\\confirmatory_evaluation.py', 'functions': ['interference', 'bootstrap', 'export_reports'], 'status': 'IMPLEMENTED in existing modules; final analysis PLANNED'}


## 12. Laporan reproducibility, tabel, plot, kesimpulan

**Status: IMPLEMENTED task map / PLANNED manuscript results.**

**Tujuan:** Memisahkan output audit aktual dari rencana eksperimen.

**Input/dependensi:** Output notebook, underlying logs, tracker, baseline hash evidence, source/config/checkpoint hashes.

**Output:** TASK 0 evidence; TASK 8 manuscript tables/plots hanya dari hasil terverifikasi.

**Kriteria penerimaan:** Simpan outputs nyata, command/timestamps/exit status; hasil gagal tetap terlihat; tidak invent execution count.

**Risiko:** Notebook saja tidak cukup sebagai bukti; snapshot penting versioned sebagai GENERATED ARTIFACT, bukan notebook primer kedua.

**Perubahan bukti ilmiah:** tidak; TASK 0 hanya membaca artefak. Implementasi/eksekusi tahap berikutnya menunggu `LANJUT TASK N`.

**Peta komponen**

| Kategori | File/artefak | Pemakaian |
|---|---|---|
| ACTIVE | `05_kode/src/classification_research.py`, `classification_pilot.py`, `frame_candidates.py`, `pilot_study.py` | Reuse training/selection; izin final belum tersedia |
| ACTIVE (draft, untracked saat audit) | `confirmatory_evaluation.py`, `confirmatory_statistics.py`, `interference_metrics.py` | Gate final, stats, interference existing |
| HISTORICAL EVIDENCE | `06_notebook/eksperimen/*.ipynb` lama, `06_notebook/hasil/`, Stage1b runs, approvals, failures | Pertahankan output, checkpoints, failed attempts |
| LEGACY IMPLEMENTATION | `bisindo_pipeline.py`, `training_pilot.py`, Detection notebooks | Primary Detection diarsipkan; helper tertentu masih direuse Classification |
| GENERATED ARTIFACT | `01_hasil/`, `03_insight_exp/`, checkpoint/log/prediction files | Selalu bedakan development/final; jangan rename/delete |
| UNKNOWN | File lain yang belum dibaca | Jangan dinyatakan dead |

**Konflik regresi yang sudah dilaporkan:** test historis `DevelopmentTests.test_notebook_has_module_calls_and_training_false`
mengharapkan outputs kosong, sedangkan notebook lama menyimpan outputs nyata. TASK 0 tidak menghapus output atau mengubah test historis.

TASK 0 tidak menjalankan ulang training, internal inference, bootstrap, data inventory lengkap, atau external scoring.
Belum ada hasil final baru. Ekspansi menambah vocabulary **dan** data/steps; tidak mengisolasi pengaruh jumlah kelas.

Peta TASK 0–8 dengan dependencies, input, file, penerimaan, command, blocker, approval ada pada tambahan TASK 0 di
`07_dokumen/TRACKING_BISINDO.md`. TASK 1 audit selesai; TASK 2–8 PLANNED/BLOCKED, tidak otomatis berjalan.


In [12]:
AUDIT_COMPLETED_UTC = datetime.now(timezone.utc).isoformat()
assert not any([RUN_TRAINING, RUN_FREEZE, RUN_EXTERNAL_SCORING])
print({'audit_started_utc': AUDIT_STARTED_UTC, 'audit_completed_utc': AUDIT_COMPLETED_UTC,
       'audit_status': 'COMPLETED safe inspection cells',
       'model_fits_this_execution': 0, 'external_scoring_this_execution': False,
       'final_freeze_this_execution': False, 'next_task': 'PLANNED TASK 1; awaiting LANJUT TASK 1'})

{'audit_started_utc': '2026-11-05T06:13:30.519102+00:00', 'audit_completed_utc': '2026-11-05T06:13:31.603238+00:00', 'audit_status': 'COMPLETED safe inspection cells', 'model_fits_this_execution': 0, 'external_scoring_this_execution': False, 'final_freeze_this_execution': False, 'next_task': 'PLANNED TASK 1; awaiting LANJUT TASK 1'}


### Status terkini setelah TASK 1

Audit TASK 1 selesai; lihat hasil sel TASK 1 dan evidence report. External readiness tetap BLOCKED.
Sel-sel TASK 0 mempertahankan output historis, termasuk pesan rencana TASK 1 pada waktu eksekusinya.
Peta task terkini mengikuti tracker existing. TASK 2 belum dimulai; perlu `LANJUT TASK 2`.
Tidak ada training, external inference, frame replacement/reselection, freeze, atau perubahan split.


### Status terkini setelah TASK 2

TASK 2 audit/usulan selesai; verifikasi aktual pada evidence validation.json dan tracker existing.
Kandidat **PROPOSED_NOT_APPROVED**; protokol tidak diamendemen. TASK 3 belum diotorisasi.
Output lama memuat status pada waktu eksekusi historis; tidak ditulis ulang.
MENUNGGU PERSETUJUAN: LANJUT TASK 3
